<a href="https://colab.research.google.com/github/oscarzgo1/AI-DEVELOPMENT-PROJECT---QSR/blob/main/QSR_13_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from qsrlib_io.world_trace import World_Trace, Object_State
from qsrlib.qsrlib import QSRlib, QSRlib_Request_Message


class QSRPipeline:

    def __init__(self, window_size=10):
        self.qsrlib = QSRlib()
        self.window_size = window_size

    # Entry Point

    def process_frames(self, collected_frames):
        """
        Main function to call from your FullPipeline
        """

        if len(collected_frames) < self.window_size:
            return None

        frames = collected_frames[-self.window_size:]

        # Keeping only consistent objects across frames
        frames = self._filter_consistent_objects(frames)

        if len(frames) < 2:
            return None

        # Build world
        world = self._build_world_trace(frames)

        # Compute QTC
        response = self._compute_qtc(world)

        # Print / Interpret
        self._print_qtc(response)

        return response


    def _filter_consistent_objects(self, frames): # Filtering consistent object in the frames
        """
        Keep only objects that appear in ALL frames
        """

        object_sets = [
            set(obj["id"] for obj in frame["objects"])
            for frame in frames
        ]

        common_ids = set.intersection(*object_sets)

        if len(common_ids) < 2:
            return []

        filtered_frames = []

        for frame in frames:
            filtered_objects = [
                obj for obj in frame["objects"]
                if obj["id"] in common_ids
            ]

            filtered_frames.append({
                "frame_id": frame["frame_id"],
                "timestamp": frame["timestamp"],
                "objects": filtered_objects
            })

        return filtered_frames


    def _build_world_trace(self, frames): # World Trace (Build)
        """
        Convert frames → QSRLib World Trace
        """

        world = World_Trace()
        object_tracks = {}

        for frame in frames:
            t = frame["timestamp"]  # use real time

            for obj in frame["objects"]:
                name = obj["id"]  # MUST be unique ID

                if name not in object_tracks:
                    object_tracks[name] = []

                object_tracks[name].append(
                    Object_State(
                        name=name,
                        timestamp=t,
                        x=obj["x"],
                        y=obj["y"]
                    )
                )

        for obj_states in object_tracks.values():
            world.add_object_state_series(obj_states)

        return world



    def _compute_qtc(self, world): # Computing QTC

        dynamic_args = {
            "qtcbs": {
                "quantisation_factor": 0.05,  # reduce noise
                "no_collapse": False
            }
        }

        request = QSRlib_Request_Message(
            which_qsr="qtcbs",
            input_data=world,
            dynamic_args=dynamic_args
        )

        return self.qsrlib.request_qsrs(request)

    # -------------------------------
    # PRINT RESULTS
    # -------------------------------
    def _print_qtc(self, response):

        print("\n--- QTC OUTPUT ---")

        for t, qsrs in response.qsrs.trace.items():
            print(f"\nTime {t}:")

            for pair, relation in qsrs.qsrs.items():
                qtc = relation.qsr
                interpretation = self._interpret_qtc(qtc)

                print(f"  {pair}: {qtc} → {interpretation}")

    # -------------------------------
    # INTERPRET QTC
    # -------------------------------
    def _interpret_qtc(self, qtc):
        """
        Simple behavior interpretation
        """

        if qtc == ('+', '+'):
            return "approaching each other"
        elif qtc == ('-', '-'):
            return "moving apart"
        elif qtc == ('+', '-'):
            return "following"
        elif qtc == ('-', '+'):
            return "moving away while other approaches"
        elif qtc == ('0', '0'):
            return "stationary"
        else:
            return "complex motion"